# Pengaruh Ukuran Input terhadap Segmentasi Instans Objek Dalam Ruangan — YOLO26s-seg

**Ringkasan.** Notebook ini melatih model pada dataset Panoptic Indoor Segmentation dengan tiga ukuran input: 320, 640, dan 960 piksel. Metrik validasi dan checkpoint terbaik disimpan; test set tidak dibaca dan inference dilakukan pada notebook terpisah.

## Tujuan dan Hipotesis

**Tujuan**
1. Mengukur pengaruh ukuran input terhadap mask mAP pada setiap model.
2. Membandingkan respons tiga keluarga arsitektur terhadap perubahan ukuran input.
3. Menyiapkan checkpoint terbaik untuk inference pada notebook testing terpisah.

**Hipotesis awal** (diuji pada bagian hasil, belum terbukti):
- **H1.** Ukuran input yang lebih besar menaikkan mask mAP, terutama pada objek kecil, dengan biaya waktu training dan inferensi yang lebih tinggi.
- **H2.** Pada split valid, kenaikan dari 640 ke 960 diperkirakan kecil. Mayoritas gambarnya berukuran 640×480, sehingga input 960 hanya memperbesar gambar tanpa menambah detail.
- **H3 (uji terpisah).** Pada test set foto sendiri (4000×3000), kenaikan dari 640 ke 960 diperkirakan lebih besar karena detail tambahan tersedia; tidak diuji di sini.

## Desain Eksperimen

| Model | Keluarga | Mekanisme mask |
|---|---|---|
| YOLO26s-seg | CNN *one-stage* | kombinasi koefisien dan *prototype mask* |
| Mask R-CNN R50-FPN v2 | CNN *two-stage* | mask biner 28×28 per RoI |
| RF-DETR-Seg Small | *Transformer* (*backbone* DINOv2) | mask per *query* |

- **Variabel bebas:** ukuran input 320, 640, dan 960 piksel. RF-DETR-Seg Small mensyaratkan kelipatan 24, sehingga memakai 312, 648, dan 960 (nilai valid terdekat).
- **Variabel kontrol (di dalam satu model):** optimizer, jumlah epoch, augmentasi, batch efektif 16, dan *seed* 42. Pelatihan multi-skala dimatikan agar ukuran input benar-benar tetap.
- **Variabel terikat:** mask mAP@[.5:.95], AP50, AP75, AR100, jam GPU, dan waktu inferensi per gambar.
- **Pemilihan checkpoint:** berdasarkan metrik validasi. Test set tidak diakses oleh notebook training; inference dilakukan di notebook terpisah.

Setelan antarmodel (epoch, augmentasi bawaan) mengikuti praktik standar tiap arsitektur. Karena itu, perbandingan utama adalah tren ukuran input di dalam satu model, bukan peringkat mutlak antarmodel.

Training berjalan paralel pada dua GPU T4 dengan antrean dinamis: GPU yang selesai lebih dulu langsung mengambil run berikutnya.

## 1. Konfigurasi

In [ ]:
from pathlib import Path
import hashlib, os, sys, json, time, shutil, subprocess
os.environ["PYTORCH_NVML_BASED_CUDA_CHECK"] = "1"  # cek GPU lewat NVML agar CUDA tidak aktif sebelum fork

DATASET_ROOT = Path("/kaggle/input/datasets/elzaff/panoptic-indoor-segmentation")  # COCO train/valid
PREVIOUS_OUTPUTS_DIR = Path("")  # gunakan hanya output tepercaya dari notebook ini untuk resume
WORK = Path("/kaggle/working")
OUT = WORK / "runs"

SIZES = [320, 640, 960]
RFDETR_RES = {320: 312, 640: 648, 960: 960}  # kelipatan 24 (patch 12 x 2 window)
MODELS = ["yolo26s-seg"]
NUM_GPUS = 2  # antrean dinamis: GPU yang kosong mengambil run berikutnya
EPOCHS = {"yolo26s-seg": 60}
BATCH = {"yolo26s-seg": {320: 16, 640: 16, 960: 8}}
EFFECTIVE_BATCH = 16  # batch x akumulasi gradien untuk model ini
SEED, WORKERS = 42, 2
WANDB_PROJECT = "indoor-instance-segmentation"
OUT.mkdir(parents=True, exist_ok=True)
print(len(MODELS) * len(SIZES), "runs")

## 2. Dependensi

In [ ]:
%pip install -q "ultralytics>=8.4.0" "pillow<13" pycocotools wandb

## 3. Runtime dan GPU

In [ ]:
import torch, torchvision, ultralytics
print({"torch": torch.__version__, "torchvision": torchvision.__version__, "ultralytics": ultralytics.__version__})
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True, check=True).stdout)
n_gpu = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())
assert n_gpu >= 2, "Aktifkan accelerator GPU T4 x2"

## 4. Dataset

**Data latih.** Panoptic Indoor Segmentation (Roboflow, format COCO *segmentation*) memakai split train dan valid bawaan tanpa pembagian ulang. Pada export yang diperiksa, 49% gambar train dan 91% gambar valid berukuran 640×480. Empat gambar (dua train, dua valid) dibuang karena ukurannya setelah koreksi EXIF tidak cocok dengan anotasi COCO, sehingga poligonnya tidak sejajar. Distribusi kelas sangat timpang: `wall-other-merged` memiliki 1.468 instans, sedangkan 16 kelas memiliki 10 instans atau kurang.

**Test set foto sendiri (untuk notebook testing terpisah; tidak dibaca di notebook ini).** Foto ruangan nyata (laboratorium, ruang kelas, kamar, kafe, toko) diambil dengan kamera ponsel pribadi, termasuk foto dalam kondisi sulit: cahaya redup, *backlight*, objek saling menutupi, dan sudut ekstrem. Anotasi disusun dalam tiga tahap:
1. Draf mask dibuat Roboflow dengan *prompt* teks per kelas dataset, satu *prompt* untuk satu kelas.
2. Tepi mask objek diperhalus Roboflow pada potongan gambar resolusi penuh. Format label disamakan dengan data latih: satu poligon per anotasi dan tanpa tumpang-tindih.
3. Seluruh mask ditinjau dan dikoreksi manual di Roboflow.

Kelas `shelf`, `wall`, `paper`, `scissor`, `st-`, `tree-merged`, dan `indoor-objects` tidak diberi *prompt*. Sebagian tidak memiliki contoh latih yang memadai, sebagian lagi duplikat kelas lain.

Sel berikut memeriksa file dan mencetak jumlah gambar serta instans hanya untuk split train dan valid.

In [ ]:
def find_root(root, split):
    """Folder yang berisi <split>/_annotations.coco.json (Kaggle kadang menambah satu folder pembungkus)."""
    if (root / split / "_annotations.coco.json").is_file():
        return root
    base = root if root.exists() else Path("/kaggle/input")  # mount path differs between Kaggle versions
    hits = [p.parent.parent for p in base.rglob("_annotations.coco.json")
            if p.parent.name == split and "own" not in str(p).lower()]
    return hits[0] if hits else None

DATASET_ROOT = find_root(DATASET_ROOT, "train")
assert DATASET_ROOT and (DATASET_ROOT / "valid/_annotations.coco.json").is_file(), "dataset train/valid tidak ditemukan"

# Pembersihan: ukuran gambar setelah koreksi EXIF harus cocok dengan anotasi.
from PIL import Image, ImageOps
CLEAN_ROOT, DROPPED = WORK / "data_clean", {}
for split in ("train", "valid"):
    src = DATASET_ROOT / split
    if not (src / "_annotations.coco.json").is_file():
        continue
    d = json.loads((src / "_annotations.coco.json").read_text(encoding="utf-8"))
    bad = set()
    for im in d["images"]:
        with Image.open(src / im["file_name"]) as f:
            if ImageOps.exif_transpose(f).size != (im["width"], im["height"]):
                bad.add(im["id"])
    DROPPED[split] = sorted(im["file_name"] for im in d["images"] if im["id"] in bad)
    d["images"] = [im for im in d["images"] if im["id"] not in bad]
    d["annotations"] = [a for a in d["annotations"] if a["image_id"] not in bad]
    dst = CLEAN_ROOT / split
    dst.mkdir(parents=True, exist_ok=True)
    for im in d["images"]:
        if not (dst / im["file_name"]).exists():
            os.symlink((src / im["file_name"]).resolve(), dst / im["file_name"])
    (dst / "_annotations.coco.json").write_text(json.dumps(d), encoding="utf-8")
print("gambar dibuang (ukuran setelah koreksi EXIF != anotasi):", {k: len(v) for k, v in DROPPED.items()}, DROPPED)
DATASET_ROOT = CLEAN_ROOT

TRAIN_COCO = json.loads((DATASET_ROOT / "train/_annotations.coco.json").read_text(encoding="utf-8"))
VALID_COCO = json.loads((DATASET_ROOT / "valid/_annotations.coco.json").read_text(encoding="utf-8"))
CATEGORY_LIST = sorted((c for c in TRAIN_COCO["categories"] if int(c["id"]) != 0), key=lambda c: int(c["id"]))
CATEGORY_IDS = [int(c["id"]) for c in CATEGORY_LIST]
CATEGORY_NAMES = [c["name"] for c in CATEGORY_LIST]
NAME_TO_ID = dict(zip(CATEGORY_NAMES, CATEGORY_IDS))
for name, d in (("train", TRAIN_COCO), ("valid", VALID_COCO)):
    print(name, "images", len(d["images"]), "instances", len(d["annotations"]))
print("foreground classes:", len(CATEGORY_NAMES))

### 4.1 Ukuran gambar dan kepadatan objek
Ukuran gambar menentukan seberapa banyak detail asli yang tersedia pada tiap ukuran input. Luas *bounding box* relatif menunjukkan proporsi objek kecil.

In [ ]:
# EDA ringkas: ukuran gambar, kepadatan objek, ukuran bbox, dan frekuensi kelas.
from collections import Counter
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

train_categories = sorted((int(c["id"]), c["name"]) for c in TRAIN_COCO["categories"])
valid_categories = sorted((int(c["id"]), c["name"]) for c in VALID_COCO["categories"])
assert train_categories == valid_categories, "Train/valid category maps differ."
assert len({int(i["id"]) for i in TRAIN_COCO["images"]}) == len(TRAIN_COCO["images"])
assert len({int(i["id"]) for i in VALID_COCO["images"]}) == len(VALID_COCO["images"])

split_data = {"train": TRAIN_COCO, "valid": VALID_COCO}
dimension_rows, object_rows, box_rows = [], [], []
counts_by_split = {}
for split, data in split_data.items():
    per_image = Counter(int(a["image_id"]) for a in data["annotations"] if int(a["category_id"]) != 0)
    counts_by_split[split] = Counter(int(a["category_id"]) for a in data["annotations"])
    for image in data["images"]:
        w, h = int(image["width"]), int(image["height"])
        dimension_rows.append({"split": split, "width": w, "height": h, "aspect_ratio": w / h})
        object_rows.append({"split": split, "instances_per_image": per_image[int(image["id"])]})
    image_by_id = {int(i["id"]): i for i in data["images"]}
    for ann in data["annotations"]:
        if int(ann["category_id"]) == 0:
            continue
        x, y, w, h = map(float, ann["bbox"])
        assert w > 0 and h > 0, f"Degenerate bbox in {split}, annotation {ann['id']}"
        im = image_by_id[int(ann["image_id"])]
        box_rows.append({"split": split, "bbox_area_fraction": w * h / (int(im["width"]) * int(im["height"]))})

dimensions = pd.DataFrame(dimension_rows)
objects_per_image = pd.DataFrame(object_rows)
bbox_fractions = pd.DataFrame(box_rows)
print("Image dimensions and aspect ratio by split:")
display(dimensions.groupby("split")[["width", "height", "aspect_ratio"]].agg(["min", "median", "max"]).round(3))
print("Instances per image:")
display(objects_per_image.groupby("split")["instances_per_image"].describe().round(2))
print("Normalized bbox area fraction:")
display(bbox_fractions.groupby("split")["bbox_area_fraction"].describe(percentiles=[.25, .5, .75]).round(4))

### 4.2 Distribusi kelas
Kelas dengan sedikit instans cenderung memiliki AP rendah dan variansnya besar. Hasil per kelas untuk kelas langka perlu dibaca hati-hati.

In [ ]:
class_rows = [{"category_id": cid, "class": name,
               "train_instances": counts_by_split["train"].get(cid, 0),
               "valid_instances": counts_by_split["valid"].get(cid, 0)}
              for cid, name in train_categories if cid != 0]
class_table = pd.DataFrame(class_rows).sort_values("train_instances", ascending=False)
print("Most frequent foreground classes (train):")
display(class_table.head(15))
print("Least frequent foreground classes (train):")
display(class_table.tail(10).sort_values("train_instances"))
ax = class_table.sort_values("train_instances").plot.barh(
    x="class", y="train_instances", figsize=(10, 16), legend=False,
    title="Train instance count by foreground class")
ax.set_xlabel("Instances"); ax.set_ylabel(""); ax.grid(axis="x", alpha=.25)
plt.tight_layout(); plt.show()

### 4.3 Pemeriksaan visual anotasi
Poligon ditumpuk di atas gambar untuk memastikan mask sesuai dengan objeknya.

In [ ]:
# Visual audit: polygon annotations over a deterministic sample from each split.
rng = np.random.default_rng(SEED)
samples = []
for split, data in split_data.items():
    per_image = Counter(int(a["image_id"]) for a in data["annotations"])
    candidates = [im for im in data["images"] if 2 <= per_image[int(im["id"])] <= 12]
    if not candidates: candidates = data["images"]
    picks = rng.choice(len(candidates), size=min(2, len(candidates)), replace=False)
    samples.extend((split, candidates[int(i)]) for i in picks)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (split, image_info) in zip(axes.flat, samples):
    image_path = DATASET_ROOT / split / image_info["file_name"]
    bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if bgr is None: raise FileNotFoundError(image_path)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    anns = [a for a in split_data[split]["annotations"]
            if int(a["image_id"]) == int(image_info["id"]) and int(a["category_id"]) != 0]
    ax.imshow(rgb)
    for j, ann in enumerate(anns):
        for polygon in ann["segmentation"]:
            points = np.asarray(polygon, dtype=float).reshape(-1, 2)
            if len(points) >= 3:
                color = plt.cm.tab20(j % 20)
                ax.add_patch(Polygon(points, closed=True, facecolor=color, edgecolor=color, alpha=.22, linewidth=1.2))
    ax.set_title(f"{split}: {image_info['file_name']} ({len(anns)} instances)", fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Pencatatan eksperimen (W&B)
Seluruh run dicatat ke W&B melalui Kaggle Secret `WANDB_API_KEY`. Sel ini menghentikan notebook bila *secret* tidak tersedia, sehingga tidak ada run yang berjalan tanpa log. Salinan log lokal tetap tersimpan di `results.csv` atau `metrics.csv`.

In [ ]:
from kaggle_secrets import UserSecretsClient
import wandb
try:
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY").strip()
except Exception as exc:
    raise RuntimeError("Tambahkan Kaggle Secret WANDB_API_KEY (Add-ons > Secrets) lalu jalankan ulang.") from exc
os.environ["WANDB_PROJECT"] = WANDB_PROJECT
assert wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True), "W&B login gagal"
print("W&B aktif:", WANDB_PROJECT)

## 6. Melanjutkan sesi
Sesi Kaggle dibatasi 12 jam. Untuk sesi baru, output versi sebelumnya dipasang dan jalurnya diisi pada `PREVIOUS_OUTPUTS_DIR`. Run yang memiliki `done.json` dilewati, sedangkan run lain dilanjutkan dari checkpoint terakhir.

In [ ]:
if str(PREVIOUS_OUTPUTS_DIR) not in ("", ".") and PREVIOUS_OUTPUTS_DIR.is_dir():
    shutil.copytree(PREVIOUS_OUTPUTS_DIR, OUT, dirs_exist_ok=True)
    print("restored", PREVIOUS_OUTPUTS_DIR)

## 7. Praproses per model

| Model | Format label | Pengubahan ukuran | Augmentasi |
|---|---|---|---|
| YOLO26s-seg | poligon ternormalisasi (txt) | *letterbox* persegi | bawaan Ultralytics: mosaic, HSV, skala, flip |
| Mask R-CNN v2 | COCO | sisi terpanjang = ukuran input | flip horizontal, kecerahan dan kontras |
| RF-DETR-Seg | COCO Roboflow | persegi tetap | bawaan RF-DETR, multi-skala dimatikan |

Augmentasi yang sama dipakai pada ketiga ukuran input dalam satu model, sehingga perbedaan hasil antarukuran tidak berasal dari augmentasi.

In [ ]:
import yaml
YOLO_ROOT = WORK / "yolo_format"
cat_to_yolo = {cid: i for i, cid in enumerate(CATEGORY_IDS)}
for split in ("train", "valid"):
    src = DATASET_ROOT / split
    raw = json.loads((src / "_annotations.coco.json").read_text(encoding="utf-8"))
    (YOLO_ROOT / split / "images").mkdir(parents=True, exist_ok=True)
    (YOLO_ROOT / split / "labels").mkdir(parents=True, exist_ok=True)
    anns = {}
    for a in raw["annotations"]:
        if int(a["category_id"]) != 0:
            anns.setdefault(int(a["image_id"]), []).append(a)
    for im in raw["images"]:
        dst = YOLO_ROOT / split / "images" / im["file_name"]
        if not dst.exists():
            os.symlink((src / im["file_name"]).resolve(), dst)
        rows = []
        for a in anns.get(int(im["id"]), []):
            for poly in a["segmentation"]:
                xy = [min(1.0, max(0.0, float(v) / (im["width"] if j % 2 == 0 else im["height"]))) for j, v in enumerate(poly)]
                if len(xy) >= 6:
                    rows.append(f"{cat_to_yolo[int(a['category_id'])]} " + " ".join(f"{v:.6f}" for v in xy))
        (YOLO_ROOT / split / "labels" / (Path(im["file_name"]).stem + ".txt")).write_text("\n".join(rows), encoding="utf-8")
YOLO_YAML = YOLO_ROOT / "dataset.yaml"
yaml.safe_dump({"path": str(YOLO_ROOT), "train": "train/images", "val": "valid/images",
                "names": dict(enumerate(CATEGORY_NAMES))}, YOLO_YAML.open("w"), sort_keys=False)

RFDETR_ROOT = WORK / "rfdetr_data"
RFDETR_ROOT.mkdir(exist_ok=True)
for split, src in (("train", DATASET_ROOT / "train"), ("valid", DATASET_ROOT / "valid")):
    if not (RFDETR_ROOT / split).exists():
        os.symlink(src.resolve(), RFDETR_ROOT / split)
print("ready:", YOLO_YAML, RFDETR_ROOT)

## 8. Konfigurasi training

| | YOLO26s-seg | Mask R-CNN v2 | RF-DETR-Seg Small |
|---|---|---|---|
| Bobot awal | COCO | COCO | COCO |
| Optimizer | AdamW, lr 1e-3, *cosine* | AdamW, lr 1e-4, *warmup* + *cosine* | AdamW, lr 1e-4 |
| Epoch | 60, *early stopping* 15 | 16 | 30, *early stopping* 10 |
| Batch (320/640/960) | 16/16/8 | 8/4/2 | 8/4/2 |
| Batch efektif | 16 | 16 | 16 |
| Presisi | AMP FP16 | AMP FP16 | AMP |

Jumlah epoch dibedakan karena laju konvergensi tiap arsitektur berbeda: Mask R-CNN yang di-*fine-tune* dari bobot COCO sudah konvergen dalam belasan epoch. Seluruh kode berada di dalam notebook. Fungsi bersama pada sel berikut dipakai oleh training Mask R-CNN dan evaluasi akhir, sehingga metrik semua model dihitung dengan kode yang sama.

In [ ]:
# Fungsi bersama: pembacaan gambar, konversi mask, evaluasi COCO, dan prediktor tiap model.
import contextlib, io, math, random, csv, traceback
import numpy as np
from PIL import Image, ImageOps
from pycocotools import mask as mu
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

METRICS = ["mask_mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR100"]


def read_json(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))


def load_rgb(path):
    with Image.open(path) as im:
        return np.asarray(ImageOps.exif_transpose(im).convert("RGB"))


def decode(seg, h, w):
    rle = mu.merge(mu.frPyObjects(seg, h, w)) if isinstance(seg, list) else (
        mu.frPyObjects(seg, h, w) if isinstance(seg["counts"], list) else seg)
    return mu.decode(rle)


def remap_gt(src_json, dst_json, n2i):
    """Tulis ulang file COCO ke ID kelas data latih (dicocokkan lewat nama). Mengembalikan kelas yang dibuang."""
    d = read_json(src_json)
    old = {int(c["id"]): c["name"] for c in d["categories"]}
    names = {old.get(int(a["category_id"])) for a in d["annotations"]}
    d["annotations"] = [dict(a, category_id=n2i[old[int(a["category_id"])]], iscrowd=int(a.get("iscrowd", 0)))
                        for a in d["annotations"] if old.get(int(a["category_id"])) in n2i]
    d["categories"] = [{"id": i, "name": n, "supercategory": "indoor"} for n, i in n2i.items()]
    Path(dst_json).write_text(json.dumps(d), encoding="utf-8")
    return sorted(n for n in names if n and n not in n2i)


def to_record(image_id, cat_id, score, mask):
    rle = mu.encode(np.asfortranarray(mask.astype(np.uint8)))
    rle["counts"] = rle["counts"].decode("ascii")
    return {"image_id": int(image_id), "category_id": int(cat_id), "segmentation": rle, "score": float(score)}


def predict_set(run, gt_json, image_root, max_det=100):
    """run(rgb) -> [(category_id, score, mask bool HxW)]; hasilnya record COCO."""
    recs = []
    for im in read_json(gt_json)["images"]:
        rgb = load_rgb(Path(image_root) / im["file_name"])
        if rgb.shape[:2] != (im["height"], im["width"]):
            raise ValueError(f"ukuran {im['file_name']} {rgb.shape[:2]} != anotasi {(im['height'], im['width'])}")
        for c, s, m in sorted(run(rgb), key=lambda t: -t[1])[:max_det]:
            if m.any():
                recs.append(to_record(im["id"], c, s, m))
    return recs


def coco_mask_eval(gt_json, records, img_ids=None):
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO(str(gt_json))
        if not records:
            return dict.fromkeys(METRICS, 0.0)
        ev = COCOeval(gt, gt.loadRes(records), "segm")
        if img_ids is not None:
            ev.params.imgIds = list(img_ids)
        ev.evaluate(); ev.accumulate(); ev.summarize()
    return dict(zip(METRICS, [float(ev.stats[i]) for i in (0, 1, 2, 3, 4, 5, 8)]))


def build_maskrcnn(num_classes, size, pretrained):
    from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
    m = maskrcnn_resnet50_fpn_v2(weights=MaskRCNN_ResNet50_FPN_V2_Weights.COCO_V1 if pretrained else None,
                                 weights_backbone=None, box_score_thresh=0.001, box_detections_per_img=100)
    m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, num_classes + 1)
    m.roi_heads.mask_predictor = MaskRCNNPredictor(m.roi_heads.mask_predictor.conv5_mask.in_channels, 256, num_classes + 1)
    m.transform.min_size, m.transform.max_size = (size,), size  # sisi terpanjang = size
    return m


def maskrcnn_runner(model, ids):
    device = next(model.parameters()).device

    def run(rgb):
        x = torch.from_numpy(rgb.copy()).permute(2, 0, 1).float().div(255).to(device)
        with torch.inference_mode(), torch.autocast(device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            o = model([x])[0]
        masks = (o["masks"][:, 0].float() > 0.5).cpu().numpy()
        return [(ids[int(l) - 1], float(s), m) for l, s, m in zip(o["labels"].tolist(), o["scores"].tolist(), masks)]
    return run


def make_predictor(kind, weights, size, n2i):
    if kind == "yolo26s-seg":
        from ultralytics import YOLO
        model = YOLO(str(weights))
        names = model.names

        def run(rgb):
            r = model.predict(rgb[..., ::-1].copy(), imgsz=size, conf=0.001, iou=0.7, max_det=100,
                              retina_masks=True, half=True, verbose=False, device=0)[0]
            if r.masks is None:
                return []
            masks = r.masks.data.cpu().numpy() > 0.5
            return [(n2i[names[int(c)]], s, m) for c, s, m in zip(r.boxes.cls.tolist(), r.boxes.conf.tolist(), masks)]
        return run
    if kind == "maskrcnn-v2":
        model = build_maskrcnn(len(n2i), size, pretrained=False)
        model.load_state_dict(torch.load(weights, map_location="cpu", weights_only=False)["model"])
        return maskrcnn_runner(model.cuda().eval(), sorted(n2i.values()))
    if kind == "rfdetr-seg-small":
        from rfdetr import RFDETRSegSmall
        model = RFDETRSegSmall(pretrain_weights=str(weights), resolution=size, trust_checkpoint=True)
        names = model.class_names

        def run(rgb):
            d = model.predict(Image.fromarray(rgb), threshold=0.001)
            if d.mask is None or len(d) == 0:
                return []
            return [(n2i[names[int(c)]], float(s), m) for c, s, m in zip(d.class_id, d.confidence, d.mask)
                    if names[int(c)] in n2i]
        return run
    raise ValueError(kind)

### 8.1 Fungsi training per model
Setiap fungsi menerima konfigurasi satu run, melanjutkan dari checkpoint terakhir bila ada, dan menulis `done.json` setelah selesai.

In [ ]:
def train_yolo(c):
    from ultralytics import YOLO, settings
    settings.update({"wandb": bool(os.environ.get("WANDB_API_KEY"))})
    out = Path(c["out"])
    last = out / "weights" / "last.pt"
    model = YOLO(str(last) if last.exists() else "yolo26s-seg.pt")
    # Augmentasi bawaan Ultralytics (mosaic, HSV, skala, flip) dan EMA tetap aktif; hanya ukuran input yang berubah.
    model.train(data=str(YOLO_YAML), epochs=c["epochs"], imgsz=c["size"], batch=c["batch"], nbs=EFFECTIVE_BATCH,
                device=0, workers=WORKERS, optimizer="AdamW", lr0=0.001, lrf=0.01, cos_lr=True, warmup_epochs=3,
                weight_decay=5e-4, patience=15, close_mosaic=10, multi_scale=0, rect=False, cache="ram", amp=True,
                seed=SEED, deterministic=False, project=str(out.parent), name=out.name, exist_ok=True,
                resume=last.exists(), plots=True, val=True)
    best = out / "weights" / "best.pt"
    assert best.exists(), "best.pt tidak ditemukan"
    (out / "done.json").write_text(json.dumps({
        "model": c["model"], "size": c["run_size"], "weights": str(best), "sha256": file_sha256(best)
    }))


class MaskRCNNTrainSet(torch.utils.data.Dataset):
    """Augmentasi sama untuk semua ukuran: flip horizontal + jitter kecerahan/kontras."""
    def __init__(self, root, label_of):
        d = read_json(Path(root) / "_annotations.coco.json")
        self.root, self.images, self.label_of, self.anns = Path(root), d["images"], label_of, {}
        for ann in d["annotations"]:
            if int(ann["category_id"]) in label_of:
                self.anns.setdefault(ann["image_id"], []).append(ann)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        from torchvision.ops import masks_to_boxes
        im = self.images[i]
        rgb = load_rgb(self.root / im["file_name"]).astype(np.float32)
        h, w = rgb.shape[:2]
        masks, labels = [], []
        for ann in self.anns.get(im["id"], []):
            m = decode(ann["segmentation"], h, w)
            if m.any():
                masks.append(m); labels.append(self.label_of[int(ann["category_id"])])
        if random.random() < 0.5:
            rgb, masks = rgb[:, ::-1], [m[:, ::-1] for m in masks]
        rgb = np.clip(rgb * random.uniform(0.7, 1.3) + random.uniform(-25, 25), 0, 255)
        masks = torch.from_numpy(np.ascontiguousarray(np.stack(masks))) if masks else torch.zeros((0, h, w), dtype=torch.uint8)
        boxes = masks_to_boxes(masks) + torch.tensor([0, 0, 1, 1]) if len(masks) else torch.zeros((0, 4))
        image = torch.from_numpy(np.ascontiguousarray(rgb)).permute(2, 0, 1).div(255)
        return image, {"boxes": boxes.float(), "labels": torch.tensor(labels, dtype=torch.int64), "masks": masks}


def collate_list(batch):
    return tuple(zip(*batch))


def train_maskrcnn(c):
    out = Path(c["out"]); out.mkdir(parents=True, exist_ok=True)
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ids = sorted(NAME_TO_ID.values())
    label_of = {cid: i + 1 for i, cid in enumerate(ids)}
    loader = torch.utils.data.DataLoader(MaskRCNNTrainSet(DATASET_ROOT / "train", label_of), batch_size=c["batch"],
                                         shuffle=True, num_workers=WORKERS, collate_fn=collate_list, drop_last=True)
    model = build_maskrcnn(len(ids), c["size"], pretrained=True).to(device)
    opt = torch.optim.AdamW([q for q in model.parameters() if q.requires_grad], lr=1e-4, weight_decay=1e-4)
    accum = max(1, EFFECTIVE_BATCH // c["batch"])
    batches_per_epoch = len(loader)
    steps_per_epoch = max(1, (batches_per_epoch + accum - 1) // accum)
    total = c["epochs"] * steps_per_epoch
    warm = max(1, total // 20)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else
                                              0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
    start, best = 0, -1.0
    if (out / "last.pt").exists():
        ck = torch.load(out / "last.pt", map_location="cpu", weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"]); start, best = ck["epoch"], ck["best"]
    run = None
    if os.environ.get("WANDB_API_KEY"):
        import wandb
        run = wandb.init(project=WANDB_PROJECT, name=f"{out.parent.name}-{out.name}", id=f"{out.parent.name}-{out.name}",
                         resume="allow", config=c)
    valid_json, valid_root = DATASET_ROOT / "valid/_annotations.coco.json", DATASET_ROOT / "valid"
    for epoch in range(start, c["epochs"]):
        t0 = time.time()
        model.train()
        sums, n = {}, 0
        for i, (images, targets) in enumerate(loader):
            images = [x.to(device, non_blocking=True) for x in images]
            targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]
            group_size = min(accum, batches_per_epoch - (i // accum) * accum)
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                losses = model(images, targets)
                loss = sum(losses.values())
            scaler.scale(loss / group_size).backward()
            if (i + 1) % accum == 0 or i + 1 == batches_per_epoch:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
                scale_before = scaler.get_scale()
                scaler.step(opt)
                scaler.update()
                if scaler.get_scale() >= scale_before:
                    sched.step()
                opt.zero_grad(set_to_none=True)
            for k, v in losses.items():
                sums[k] = sums.get(k, 0.0) + float(v)
            sums["loss"] = sums.get("loss", 0.0) + float(loss); n += 1
        model.eval()
        metrics = coco_mask_eval(valid_json, predict_set(maskrcnn_runner(model, ids), valid_json, valid_root))
        row = {"epoch": epoch + 1, **{f"train/{k}": v / max(n, 1) for k, v in sums.items()},
               **{f"val/{k}": v for k, v in metrics.items()}, "lr": opt.param_groups[0]["lr"], "epoch_seconds": time.time() - t0}
        new = not (out / "results.csv").exists()
        with (out / "results.csv").open("a", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(row))
            if new: w.writeheader()
            w.writerow(row)
        if run: run.log(row, step=epoch + 1)
        print(f"epoch {epoch + 1}/{c['epochs']} loss={row['train/loss']:.3f} val mask_mAP={metrics['mask_mAP']:.4f}", flush=True)
        if metrics["mask_mAP"] > best:
            best = metrics["mask_mAP"]
            torch.save({"model": model.state_dict(), "epoch": epoch + 1, "val": metrics}, out / "best.pt")
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "epoch": epoch + 1, "best": best}, out / "last.pt")
    if run: run.finish()
    best_path = out / "best.pt"
    (out / "done.json").write_text(json.dumps({
        "model": c["model"], "size": c["run_size"], "weights": str(best_path),
        "sha256": file_sha256(best_path), "best_val_mask_mAP": best
    }))


def train_rfdetr(c):
    from rfdetr import RFDETRSegSmall
    out = Path(c["out"]); out.mkdir(parents=True, exist_ok=True)
    ckpts = [p for p in out.rglob("*") if p.suffix in (".ckpt", ".pth") and "best" not in p.name]
    resume = str(max(ckpts, key=lambda p: p.stat().st_mtime)) if ckpts else None
    model = RFDETRSegSmall(resolution=c["size"])
    model.train(dataset_dir=str(RFDETR_ROOT), output_dir=str(out), epochs=c["epochs"], batch_size=c["batch"],
                grad_accum_steps=max(1, EFFECTIVE_BATCH // c["batch"]), lr=1e-4, num_workers=WORKERS, seed=SEED,
                multi_scale=False, expanded_scales=False,  # ukuran input tetap: variabel yang diteliti
                early_stopping=True, early_stopping_patience=10, checkpoint_interval=5, resume=resume,
                tensorboard=True, wandb=bool(os.environ.get("WANDB_API_KEY")), project=WANDB_PROJECT,
                run=f"{out.parent.name}-{out.name}", progress_bar=None)
    best = sorted((p for p in out.rglob("*best*") if p.suffix in (".ckpt", ".pth", ".pt")),
                  key=lambda p: ("total" not in p.name, -p.stat().st_mtime))
    assert best, f"checkpoint best tidak ditemukan di {out}"
    (out / "done.json").write_text(json.dumps({
        "model": c["model"], "size": c["run_size"], "weights": str(best[0]),
        "sha256": file_sha256(best[0]), "candidates": [str(p) for p in best]
    }))


TRAINERS = {"yolo26s-seg": train_yolo, "maskrcnn-v2": train_maskrcnn, "rfdetr-seg-small": train_rfdetr}

## 9. Pelaksanaan training
Tiga run untuk ukuran input 320, 640, dan 960 pada model ini dibagikan lewat antrean dinamis ke dua GPU. GPU yang kosong mengambil run tertunda berikutnya, dimulai dari ukuran terbesar agar beban seimbang. Run yang sudah memiliki `done.json` dilewati untuk mendukung resume. Setiap run berjalan di proses anak dengan satu GPU; log tersimpan di `runs/size_*/<model>/train.log`.

In [ ]:
import multiprocessing as mp
from multiprocessing.connection import wait as wait_processes


def run_dir(model, size):
    return OUT / f"size_{size}" / model
CHECKPOINT_CACHE = {}


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def completed_checkpoint(model, size):
    done = run_dir(model, size) / "done.json"
    try:
        record = json.loads(done.read_text())
        if record.get("model") != model or record.get("size") != size:
            return None
        expected = record.get("sha256")
        if not isinstance(expected, str) or len(expected) != 64:
            return None
        weights = Path(record["weights"]).resolve()
        weights.relative_to(run_dir(model, size).resolve())
        if not weights.is_file():
            return None
        stat = weights.stat()
        if stat.st_size == 0:
            return None
        signature = (stat.st_ino, stat.st_size, stat.st_mtime_ns, stat.st_ctime_ns, expected)
        if CHECKPOINT_CACHE.get(weights) != signature:
            if file_sha256(weights) != expected:
                return None
            CHECKPOINT_CACHE[weights] = signature
        return weights
    except (OSError, ValueError, KeyError, TypeError, AttributeError):
        return None

def train_size(model, size):
    return RFDETR_RES[size] if model.startswith("rfdetr") else size


def _child(model, size, gpu, log_path):
    """Proses anak: kunci satu GPU, alihkan stdout/stderr ke file log, lalu jalankan training."""
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu)
    fd = os.open(log_path, os.O_WRONLY | os.O_CREAT | os.O_APPEND)
    os.dup2(fd, 1); os.dup2(fd, 2)
    sys.stdout = open(1, "w", buffering=1, closefd=False)
    sys.stderr = open(2, "w", buffering=1, closefd=False)
    code = 0
    try:
        TRAINERS[model]({"model": model, "size": train_size(model, size), "run_size": size,
                         "epochs": EPOCHS[model], "batch": BATCH[model][size], "out": str(run_dir(model, size))})
    except BaseException:
        traceback.print_exc(); code = 1
    sys.stdout.flush(); sys.stderr.flush()
    os._exit(code)


def launch(model, size, gpu):
    d = run_dir(model, size)
    if completed_checkpoint(model, size) is not None:
        print("skip", model, size); return None
    d.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    p = PROCESS_CONTEXT.Process(target=_child, args=(model, size, gpu, str(d / "train.log")))
    p.start()
    return p, t0


def finish_run(p, model, size, gpu, t0):
    p.join()
    rc, secs = p.exitcode, time.time() - t0
    if rc == 0 and completed_checkpoint(model, size) is None:
        rc = 1
        print(f"FAIL {model} {size}: done.json/checkpoint valid tidak ditemukan", flush=True)
    try:
        new = not (OUT / "gpu_time.csv").exists()
        with (OUT / "gpu_time.csv").open("a") as f:
            if new: f.write("model,size,gpu,started_unix,seconds,returncode\n")
            f.write(f"{model},{size},{gpu},{t0:.0f},{secs:.0f},{rc}\n")
        with (OUT / "LOGBOOK.md").open("a") as f:
            f.write(f"- {time.strftime('%Y-%m-%d %H:%M')} | {model} size={size} gpu={gpu} rc={rc} {secs / 3600:.2f} h\n")
        print(f"{'OK  ' if rc == 0 else 'FAIL'} {model} {size}: {secs / 60:.1f} min", flush=True)
        if rc != 0:
            log = run_dir(model, size) / "train.log"
            if log.exists():
                print("".join(log.read_text(errors="replace").splitlines(True)[-20:]), flush=True)
    finally:
        p.close()
    return rc == 0


# Run terpanjang didahulukan; scheduler utama menjaga maksimal dua proses GPU aktif.
PENDING = [(m, s) for s in sorted(SIZES, reverse=True) for m in MODELS if completed_checkpoint(m, s) is None]
FAILURES = []
print("run tertunda:", PENDING)

# CUDA tidak boleh aktif di proses utama sebelum fork; jumlah GPU sudah diperiksa lewat nvidia-smi.
assert not torch.cuda.is_initialized(), "Restart kernel: CUDA sudah aktif di proses utama sebelum fork."
PROCESS_CONTEXT = mp.get_context("fork")
FREE_GPUS = list(range(NUM_GPUS))
RUNNING = {}

while PENDING or RUNNING:
    while PENDING and FREE_GPUS:
        model, size = PENDING.pop(0)
        gpu = FREE_GPUS.pop(0)
        try:
            started = launch(model, size, gpu)
            if started is None:
                FREE_GPUS.append(gpu)
                continue
            process, t0 = started
            RUNNING[process.sentinel] = (process, model, size, gpu, t0)
        except Exception as exc:
            failure = f"{model} size={size} gpu={gpu}: gagal diluncurkan ({exc!r})"
            FAILURES.append(failure)
            FREE_GPUS.append(gpu)
            print("FAIL", failure, flush=True)

    if not RUNNING:
        break
    for sentinel in wait_processes(list(RUNNING)):
        process, model, size, gpu, t0 = RUNNING.pop(sentinel)
        try:
            if not finish_run(process, model, size, gpu, t0):
                FAILURES.append(f"{model} size={size} gpu={gpu}: child gagal, periksa train.log")
        except Exception as exc:
            failure = f"{model} size={size} gpu={gpu}: gagal menyelesaikan run ({exc!r})"
            FAILURES.append(failure)
            print("FAIL", failure, flush=True)
        finally:
            FREE_GPUS.append(gpu)

if FAILURES:
    raise RuntimeError("Training gagal: " + "; ".join(FAILURES))

## 10. Evaluasi

Prediksi split valid dikonversi ke COCO RLE dan dinilai dengan `pycocotools` (mode *segm*). Notebook ini hanya mengevaluasi valid; test/inference dilakukan pada notebook terpisah.

- **mask mAP@[.5:.95]**: rata-rata AP pada ambang IoU 0,50 sampai 0,95. Ini metrik utama.
- **AP50 / AP75**: AP pada IoU 0,50 dan 0,75. AP75 lebih peka terhadap ketepatan tepi mask.
- **AR100**: *recall* rata-rata dengan maksimal 100 deteksi per gambar.
- **Waktu inferensi**: detik per gambar, termasuk praproses dan konversi mask.

In [ ]:
import pandas as pd

EVAL = WORK / "eval"; EVAL.mkdir(exist_ok=True)
SETS = {"valid": DATASET_ROOT / "valid"}
GT = {}
for name, folder in SETS.items():
    GT[name] = EVAL / f"gt_{name}.json"
    print(name, "classes not in training set (ignored):", remap_gt(folder / "_annotations.coco.json", GT[name], NAME_TO_ID))

rows = []
for size in SIZES:
    for model in MODELS:
        weights = completed_checkpoint(model, size)
        if weights is None:
            raise FileNotFoundError(f"checkpoint valid tidak ditemukan untuk {model} size={size}")
        run = make_predictor(model, str(weights), train_size(model, size), NAME_TO_ID)
        for name, folder in SETS.items():
            t0 = time.time()
            recs = predict_set(run, GT[name], folder)
            (EVAL / f"pred_{model}_{size}_{name}.json").write_text(json.dumps(recs))
            n_img = len(json.loads(GT[name].read_text())["images"])
            rows.append({"model": model, "size": size, "set": name, **coco_mask_eval(GT[name], recs),
                         "sec_per_img": (time.time() - t0) / n_img})
            print(rows[-1], flush=True)
        del run; torch.cuda.empty_cache()
RESULTS = pd.DataFrame(rows)
RESULTS.to_csv(OUT / "final_eval.csv", index=False)
display(RESULTS.pivot_table(index=["model", "size"], columns="set", values="mask_mAP").round(4))

### 10.1 Mask mAP valid terhadap ukuran input dan biaya GPU
Grafik membandingkan mAP valid pada tiga ukuran input dan biaya GPU; tidak ada split test yang dibaca atau dievaluasi.

In [ ]:
sets = list(SETS)
fig, axes = plt.subplots(1, len(sets), figsize=(5 * len(sets), 4), sharey=True, squeeze=False)
for ax, name in zip(axes[0], sets):
    for model, g in RESULTS[RESULTS.set == name].groupby("model"):
        ax.plot(g["size"], g["mask_mAP"], marker="o", label=model)
    ax.set_title(name); ax.set_xlabel("input size"); ax.set_xticks(SIZES); ax.grid(alpha=.3)
axes[0][0].set_ylabel("mask mAP@[.5:.95]"); axes[0][0].legend()
plt.tight_layout(); plt.savefig(OUT / "map_vs_size.png", dpi=150); plt.show()


gpu = pd.read_csv(OUT / "gpu_time.csv").groupby(["model", "size"], as_index=False)["seconds"].sum()
gpu["gpu_hours"] = gpu["seconds"] / 3600
display(gpu.round(3)); gpu.to_csv(OUT / "gpu_hours.csv", index=False)

### 10.2 Temuan utama
Sel berikut menyusun temuan langsung dari `final_eval.csv`, sehingga narasi selalu sesuai dengan angka hasil run.

In [ ]:
from IPython.display import Markdown
lines = []
for model, g in RESULTS.groupby("model"):
    for name, h in g.groupby("set"):
        m = h.set_index("size")["mask_mAP"]
        best = m.idxmax()
        d = m.get(960, np.nan) - m.get(640, np.nan)
        t = h.set_index("size")["sec_per_img"]
        lines.append(f"- **{model}** pada *{name}*: mAP tertinggi pada {best} px ({m[best]:.3f}); "
                     f"selisih 960 − 640 = {d:+.3f}; waktu inferensi {t.min():.2f}–{t.max():.2f} s/gambar.")
display(Markdown("\n".join(lines)))

## 11. Analisis kegagalan

Lima gambar valid dengan AP per gambar terendah ditampilkan berdampingan dengan *ground truth*. Setiap kegagalan dikaitkan dengan mekanisme model:
- **YOLO26s-seg**: mask disusun dari *prototype* beresolusi 1/4 input, sehingga tepi objek tipis atau kecil mudah kabur.
- **Mask R-CNN v2**: mask setiap objek diprediksi pada grid 28×28 lalu diperbesar, sehingga detail objek besar ikut hilang.
- **RF-DETR-Seg**: mask berasal dari *query* dan fitur *patch* 12 piksel, sehingga objek yang lebih kecil dari beberapa *patch* sulit dipisahkan.

Model dan ukuran yang dianalisis diatur melalui `FAIL_MODEL` dan `FAIL_SIZE`.

In [ ]:
FAIL_MODEL, FAIL_SIZE = MODELS[0], 640
FAIL_SET = "valid"
gt = read_json(GT[FAIL_SET])
recs = json.loads((EVAL / f"pred_{FAIL_MODEL}_{FAIL_SIZE}_{FAIL_SET}.json").read_text())
per_img = [(im, coco_mask_eval(GT[FAIL_SET], [r for r in recs if r["image_id"] == im["id"]], [im["id"]])["mask_mAP"])
           for im in gt["images"]]
worst = sorted(per_img, key=lambda t: t[1])[:5]
fig, axes = plt.subplots(len(worst), 2, figsize=(14, 5 * len(worst)), squeeze=False)
for (im, ap), (a0, a1) in zip(worst, axes):
    rgb = load_rgb(SETS[FAIL_SET] / im["file_name"])
    panels = ((a0, [decode(a["segmentation"], im["height"], im["width"]) for a in gt["annotations"] if a["image_id"] == im["id"]], "GT"),
              (a1, [decode(r["segmentation"], im["height"], im["width"]) for r in recs if r["image_id"] == im["id"] and r["score"] >= 0.25], f"pred@0.25 (AP={ap:.3f})"))
    for ax, masks, title in panels:
        over = rgb.copy()
        for k, m in enumerate(masks):
            sel = m.astype(bool)
            over[sel] = (0.5 * over[sel] + 0.5 * np.array(plt.cm.tab20(k % 20)[:3]) * 255).astype(np.uint8)
        ax.imshow(over); ax.set_title(f"{im['file_name']} | {title} | {len(masks)} masks"); ax.axis("off")
plt.tight_layout(); plt.savefig(OUT / f"failures_{FAIL_MODEL}_{FAIL_SIZE}.png", dpi=110); plt.show()

### 11.1 Uji penjelasan: kegagalan yang sama pada ukuran input lain

Bila kegagalan disebabkan resolusi (objek terlalu kecil atau tepi kabur), AP pada foto valid yang sama seharusnya naik ketika ukuran input dinaikkan. Bila tetap rendah pada semua ukuran, penyebabnya lebih mungkin kelas yang jarang muncul di data latih atau kualitas anotasi; analisis *domain shift* ditunda ke notebook testing.

In [ ]:
preds_by_size = {s: json.loads(p.read_text()) for s in SIZES
                 if (p := EVAL / f"pred_{FAIL_MODEL}_{s}_{FAIL_SET}.json").exists()}
rows = []
for im, _ in worst:
    row = {"file": im["file_name"]}
    for s, rs in preds_by_size.items():
        row[f"AP@{s}"] = coco_mask_eval(GT[FAIL_SET], [r for r in rs if r["image_id"] == im["id"]], [im["id"]])["mask_mAP"]
    rows.append(row)
display(pd.DataFrame(rows).round(3))

## 12. Keterbatasan

- Test set foto sendiri dianotasi di Roboflow dengan penyesuaian label terhadap 74 kelas pelatihan; data uji tidak pernah dipakai saat training.
- RF-DETR memakai 312 dan 648 piksel, bukan tepat 320 dan 640.
- Empat gambar dataset asli (dua train, dua valid) dibuang karena anotasinya tidak sejajar dengan gambar; metrik valid dihitung pada 136 gambar.
- Setiap konfigurasi dijalankan dengan satu *seed*, sehingga variasi antar-*seed* belum terukur.
- Catatan untuk notebook testing terpisah: pada foto 4000×3000, hampir semua objek masuk kategori COCO *large* (>96² piksel), sehingga AP_small dan AP_medium tidak sebanding antar-set.
- Sebanyak 17 foto test diterima melalui WhatsApp, yang menghapus EXIF. EXIF foto tersebut direkonstruksi dengan izin asisten dosen: tanggal diambil dari nama file, sedangkan jam merupakan perkiraan. Setiap file rekonstruksi diberi penanda pada kolom `ImageDescription`.

## 13. Artefak

| Berkas | Isi |
|---|---|
| `runs/size_*/<model>/results.csv`, `metrics.csv` | log training asli per epoch |
| `runs/gpu_time.csv`, `runs/gpu_hours.csv` | catatan waktu GPU per run |
| `runs/LOGBOOK.md` | kronologi run; kendala dan keputusan ditambahkan manual |
| `runs/final_eval.csv` | metrik akhir pada split valid |
| `runs/map_vs_size.png`, `runs/failures_*.png` | grafik dan visualisasi kegagalan |

## 14. Deklarasi Penggunaan AI

Claude (Anthropic) dipakai untuk menyusun notebook, *pipeline* pelabelan Roboflow + Roboflow, dan skrip rekonstruksi EXIF. Seluruh keluaran AI diperiksa dan diuji sebelum dipakai.

**Prompt utama** (diringkas):
1. Menilai kesesuaian notebook dengan ketentuan tugas dan memberi saran training yang cepat tetapi hasilnya baik.
2. Menambahkan model *transformer* sebagai keluarga arsitektur ketiga.
3. Merapikan hasil pelabelan Roboflow agar sesuai dengan kelas dataset.

**Kasus saran AI yang keliru dan perbaikannya:**
- Test set foto sendiri dianotasi di Roboflow dengan penyesuaian label terhadap 74 kelas pelatihan; data uji tidak pernah dipakai saat training.
- Skrip rekonstruksi EXIF awal menghasilkan jam pengambilan foto yang lebih lambat daripada waktu foto dikirim. Perbaikan: rentang jam dibatasi pada pagi hari.
- Notebook versi awal melatih YOLO dengan batch 1, yang membuat statistik *BatchNorm* tidak stabil. Perbaikan: batch 8–16 dengan batch efektif 16.

## 15. Kesimpulan

*Diisi setelah run selesai, berdasarkan Bagian 10.2 dan 11.1. Setiap poin menjawab satu tujuan.*
1. **Ukuran input:** apakah H1 terbukti, dan pada model mana kenaikan 640 → 960 paling besar?
2. **Keluarga arsitektur:** model mana yang paling peka dan paling stabil terhadap perubahan ukuran input?
3. **Test set:** domain shift akan diukur di notebook testing terpisah; test tidak dijalankan di sini.